In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [45]:
data = pd.read_csv("../../Data/filtered_data.csv", low_memory=False)

data["Last Week"] = pd.to_numeric(data["Last Week"], errors='coerce').astype('float64')

data.head()
print('Last Week dtype:', data['Last Week'].dtype)


Last Week dtype: float64


In [ ]:
# Create rank_change feature
# Make an exception for the first week, where there is no "Last Week" value, set rank_change to NaN and then fill it with 0
#data.loc[data["Weeks in Charts"] == "NaN", "rank_change"] = np.nan
data.loc[data["Weeks in Charts"] != "NaN", "rank_change"] = data["Last Week"] - data["Rank"]

In [57]:
data.head()

,Date,Song,Artist,Rank,Last Week,Peak Position,Weeks in Charts,Normalized Title,Track,Danceability,...,Liveness,Valence,Tempo,Duration_ms,Time_signature,Chorus_hit,Sections,Target,rank_change,target_next_rank
0,1958-12-17,Jingle Bell Rock,Bobby Helms,57,NaN,1,NaN,jingle bell rock,Jingle Bell Rock,0.754,...,0.0652,0.806,119.705,130973,4,38.40391,7,1,NaN,35.0
1,1958-12-24,Jingle Bell Rock,Bobby Helms,35,57.0,35,2.0,jingle bell rock,Jingle Bell Rock,0.754,...,0.0652,0.806,119.705,130973,4,38.40391,7,1,22.0,45.0
2,1958-12-31,Jingle Bell Rock,Bobby Helms,45,35.0,35,3.0,jingle bell rock,Jingle Bell Rock,0.754,...,0.0652,0.806,119.705,130973,4,38.40391,7,1,-10.0,70.0
3,1959-01-07,Jingle Bell Rock,Bobby Helms,70,45.0,35,4.0,jingle bell rock,Jingle Bell Rock,0.754,...,0.0652,0.806,119.705,130973,4,38.40391,7,1,-25.0,69.0
4,1960-12-07,Rockin' Around The Christmas Tree,Brenda Lee,64,NaN,1,NaN,rockin around the christmas tree,Rockin' Around The Christmas Tree,0.589,...,0.5050,0.898,67.196,126267,4,33.91817,6,1,NaN,26.0


In [ ]:
# Create the target variable "target_next_rank" by shifting the "Rank" column up by one week for each song
data['target_next_rank'] = (
    data.groupby('Song')['Rank']
      .shift(-1)
)

In [60]:
data[data['Track'] == 'Blinding Lights']

,Date,Song,Artist,Rank,Last Week,Peak Position,Weeks in Charts,Normalized Title,Track,Danceability,...,Time_signature,Chorus_hit,Sections,Target,rank_change,target_next_rank,rolling_avg_4_weeks,rolling_avg_all_time,rolling_std_4_weeks,rolling_std_all_time
138272,2019-12-11,Blinding Lights,The Weeknd,11,NaN,1,NaN,blinding lights,Blinding Lights,0.513,...,4,33.1221,9,1,NaN,52.0,11.00,11.000000,NaN,NaN
138394,2019-12-18,Blinding Lights,The Weeknd,52,11.0,11,2.0,blinding lights,Blinding Lights,0.513,...,4,33.1221,9,1,-41.0,63.0,31.50,31.500000,28.991378,28.991378
138489,2019-12-25,Blinding Lights,The Weeknd,63,52.0,11,3.0,blinding lights,Blinding Lights,0.513,...,4,33.1221,9,1,-11.0,72.0,42.00,42.000000,27.404379,27.404379
138567,2020-01-01,Blinding Lights,The Weeknd,72,63.0,11,4.0,blinding lights,Blinding Lights,0.513,...,4,33.1221,9,1,-9.0,59.0,49.50,49.500000,26.938201,26.938201
138638,2020-01-08,Blinding Lights,The Weeknd,59,72.0,11,5.0,blinding lights,Blinding Lights,0.513,...,4,33.1221,9,1,13.0,39.0,61.50,51.400000,8.346656,23.712866
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
139683,2021-07-28,Blinding Lights,The Weeknd,17,17.0,1,85.0,blinding lights,Blinding Lights,0.513,...,4,33.1221,9,1,0.0,17.0,16.75,10.694118,1.258306,13.394227
139684,2021-08-04,Blinding Lights,The Weeknd,17,17.0,1,86.0,blinding lights,Blinding Lights,0.513,...,4,33.1221,9,1,0.0,16.0,16.50,10.767442,1.000000,13.332556
139685,2021-08-11,Blinding Lights,The Weeknd,16,17.0,1,87.0,blinding lights,Blinding Lights,0.513,...,4,33.1221,9,1,1.0,18.0,16.75,10.827586,0.500000,13.266681
139686,2021-08-18,Blinding Lights,The Weeknd,18,16.0,1,88.0,blinding lights,Blinding Lights,0.513,...,4,33.1221,9,1,-2.0,21.0,17.00,10.909091,0.816497,13.212356


In [ ]:
data = data.dropna(subset=['target_next_rank']) #drop rows where target_next_rank is NaN, which means it's the last week of the song in the charts and we don't have a target value for it

In [58]:
# Create rolling average features for the past 4 weeks and all time with numpy
data['rolling_avg_4_weeks'] = data.groupby('Song')['Rank'].transform(lambda x: x.rolling(window=4, min_periods=1).mean())
data['rolling_avg_all_time'] = data.groupby('Song')['Rank'].transform(lambda x: x.expanding(min_periods=1).mean())
data['rolling_std_4_weeks'] = data.groupby('Song')['Rank'].transform(lambda x: x.rolling(window=4, min_periods=1).std())
data['rolling_std_all_time'] = data.groupby('Song')['Rank'].transform(lambda x: x.expanding(min_periods=1).std())


In [67]:
data.head()

,Date,Song,Artist,Rank,Last Week,Weeks in Charts,Normalized Title,Danceability,Energy,Key,...,Duration_ms,Time_signature,Chorus_hit,Sections,rank_change,target_next_rank,rolling_avg_4_weeks,rolling_avg_all_time,rolling_std_4_weeks,rolling_std_all_time
0,1958-12-17,Jingle Bell Rock,Bobby Helms,57,NaN,NaN,jingle bell rock,0.754,0.424,2,...,130973,4,38.40391,7,NaN,35.0,57.000000,57.000000,NaN,NaN
1,1958-12-24,Jingle Bell Rock,Bobby Helms,35,57.0,2.0,jingle bell rock,0.754,0.424,2,...,130973,4,38.40391,7,22.0,45.0,46.000000,46.000000,15.556349,15.556349
2,1958-12-31,Jingle Bell Rock,Bobby Helms,45,35.0,3.0,jingle bell rock,0.754,0.424,2,...,130973,4,38.40391,7,-10.0,70.0,45.666667,45.666667,11.015141,11.015141
3,1959-01-07,Jingle Bell Rock,Bobby Helms,70,45.0,4.0,jingle bell rock,0.754,0.424,2,...,130973,4,38.40391,7,-25.0,69.0,51.750000,51.750000,15.129992,15.129992
4,1960-12-07,Rockin' Around The Christmas Tree,Brenda Lee,64,NaN,NaN,rockin around the christmas tree,0.589,0.472,8,...,126267,4,33.91817,6,NaN,26.0,64.000000,64.000000,NaN,NaN


In [ ]:
# Drop Peak Position as it's not a feature we would have in a real forecasting scenario, as we won't know the peak position of a song until it leaves the charts, which is after our forecasting period
#data = data.drop(columns=['Peak Position'])
# Drop Track column as it's just the song name and we already have the Song column
#data = data.drop(columns=['Track'])
# Drop Target column as it is a variable from previous classification dataset
#data = data.drop(columns=['Target'])

In [68]:
data.to_csv("../../Data/processed_forecast_data.csv", index=False)